# 01 · The free 100-ETF universe

**Block 2-3 of the workshop agenda** ("From Market Idea to Alpha Factor" /
"The Modern Quant Stack") — everything downstream in this workshop runs on
this dataset, so we start by loading it and confirming what it actually
contains before trusting anything built on top of it.

The data ships with this repo (`data/etf_universe.parquet`) so the session
doesn't depend on live internet access. It's the same free, Yahoo-Finance
-sourced universe used in the ETF case study in *Machine Learning for
Trading, 3rd Edition* — 100 liquid ETFs, selected backward-looking on a
$10M average-daily-volume threshold (see `data/README.md` for the point
-in-time eligibility file and its caveats).

In [1]:
import pandas as pd
import polars as pl

DATA_DIR = "../data"

prices = pd.read_parquet(f"{DATA_DIR}/etf_universe.parquet")
prices["timestamp"] = pd.to_datetime(prices["timestamp"])
prices = prices.sort_values(["symbol", "timestamp"]).reset_index(drop=True)
prices.head()

,timestamp,open,high,low,close,volume,symbol
0,2008-03-28,35.668968,35.668968,35.668968,35.668968,200.0,ACWI
1,2008-03-31,35.597774,35.597774,35.113644,35.113644,400.0,ACWI
2,2008-04-01,35.775767,36.060551,35.711691,36.060551,600.0,ACWI
3,2008-04-02,36.395160,36.594509,36.345323,36.459236,10700.0,ACWI
4,2008-04-03,42.503738,42.503738,36.274128,36.665703,29100.0,ACWI


## Sanity checks before we build anything

A workflow starts by interrogating the data, not by trusting the file
name. Three questions: how many names, what date range, how complete.

In [2]:
print(f"{prices['symbol'].nunique()} symbols")
print(f"{prices['timestamp'].min().date()} -> {prices['timestamp'].max().date()}")

coverage = prices.groupby("symbol")["timestamp"].agg(["min", "max", "count"])
coverage["years"] = ((coverage["max"] - coverage["min"]).dt.days / 365.25).round(1)
coverage.sort_values("min").head(10)

100 symbols
2006-01-03 -> 2025-12-31


,min,max,count,years
symbol,,,,
IYR,2006-01-03,2025-12-31,5031,20.0
SHY,2006-01-03,2025-12-31,5031,20.0
GLD,2006-01-03,2025-12-31,5031,20.0
XLB,2006-01-03,2025-12-31,5031,20.0
VWO,2006-01-03,2025-12-31,5031,20.0
IAU,2006-01-03,2025-12-31,5031,20.0
IBB,2006-01-03,2025-12-31,5031,20.0
IEF,2006-01-03,2025-12-31,5031,20.0
VUG,2006-01-03,2025-12-31,5031,20.0


Coverage is **not uniform** — some ETFs (SPY, QQQ) have traded since the
1990s/2000s; others launched much later (XLC in 2018, sector-momentum
funds like MTUM/VLUE in 2013). `coverage["min"]` shows this directly.
This matters the moment you rank across the full universe on any given
date: an equal-weight backtest starting in 2006 trades a very different,
much smaller universe than one starting in 2020.

In [3]:
short_history = coverage[coverage["years"] < 10].sort_values("years")
print(f"{len(short_history)} of {len(coverage)} ETFs have under 10 years of history")
short_history.head(10)

1 of 100 ETFs have under 10 years of history


,min,max,count,years
symbol,,,,
XLC,2018-06-19,2025-12-31,1895,7.5


## Point-in-time eligibility

`data/eligibility.csv` records, for each (symbol, year), whether that ETF
passed the $10M-ADV eligibility bar *as of that year* — this is what a
point-in-time backtest must filter on, not "is this symbol in the file at
all." The full 100-symbol universe was selected **backward-looking**, so
using all 100 from day one of any backtest is itself a survivorship-bias
bug — the same bug documented for the book's own US-equities case study
(Ch2 errata). Treat `eligibility.csv` as required reading before wiring
any of this into a backtest, not an optional file.

In [4]:
eligibility = pd.read_csv(f"{DATA_DIR}/eligibility.csv")
eligible_per_year = eligibility.groupby("eligible_year")["symbol"].nunique()
eligible_per_year

eligible_year
2007    43
2008    53
2009    67
2010    67
2011    75
2012    78
2013    78
2014    84
2015    89
2016    88
2017    94
2018    96
2019    96
2020    95
2021    95
2022    95
2023    96
2024    93
2025    93
2026    95
Name: symbol, dtype: int64

## From pandas to polars

`ml4t-backtest` is polars-first. We keep pandas for exploration (it's
what most of the room already knows) and convert once, right before it
touches the engine — that boundary is worth being deliberate about rather
than mixing the two libraries ad hoc through the pipeline.

In [5]:
prices_pl = pl.from_pandas(prices)
prices_pl.head()

timestamp,open,high,low,close,volume,symbol
datetime[μs],f64,f64,f64,f64,f64,str
2008-03-28 00:00:00,35.668968,35.668968,35.668968,35.668968,200.0,"""ACWI"""
2008-03-31 00:00:00,35.597774,35.597774,35.113644,35.113644,400.0,"""ACWI"""
2008-04-01 00:00:00,35.775767,36.060551,35.711691,36.060551,600.0,"""ACWI"""
2008-04-02 00:00:00,36.39516,36.594509,36.345323,36.459236,10700.0,"""ACWI"""
2008-04-03 00:00:00,42.503738,42.503738,36.274128,36.665703,29100.0,"""ACWI"""


**Next:** `02_features_labels.ipynb` — turn this raw OHLCV panel into
momentum/volatility/RSI features and forward-return labels.